# 07 — Interpretabilidad (LIME + SHAP)

Explicaciones locales y sonda de sesgo dialectal.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from discurso_odio.config import load_settings
from discurso_odio.data.loaders import load_all_splits, load_split, get_class_weights
from discurso_odio.paths import REPORTS_FIGURES, REPORTS_METRICS
from discurso_odio.utils.plotting import setup_plot_style, get_label_names, get_class_palette
from discurso_odio.utils.seed import set_seed

settings = load_settings()
set_seed(settings['project']['seed'])
setup_plot_style()
LABELS = get_label_names()
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS.mkdir(parents=True, exist_ok=True)
print('Proyecto:', settings['project']['name'])

from functools import partial
from discurso_odio.interpretability.lime_explainer import (
    build_lime_explainer, make_predict_proba_fn, explain_instance,
    aggregate_lime_weights, jaccard_top_k,
)
from discurso_odio.interpretability.shap_explainer import (
    make_shap_predict_fn, explain_texts_shap, extract_top_tokens,
)
from discurso_odio.models.mbert import predict, get_device, build_model, build_tokenizer
from discurso_odio.paths import MODELS_DIR
from transformers import AutoModelForSequenceClassification
import torch

# Predictor en memoria (mismo patrón que scripts/run_interpretability.py)
class CachedPredictor:
    def __init__(self, model_dir):
        settings = load_settings()
        self.device = get_device()
        self.tokenizer = build_tokenizer()
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.model.to(self.device)
        self.model.eval()
        self.max_length = settings['model']['max_length']

    @torch.no_grad()
    def predict_proba(self, texts):
        import torch
        texts = [str(t) if t else ' ' for t in texts]
        enc = self.tokenizer(texts, truncation=True, padding=True,
                             max_length=self.max_length, return_tensors='pt')
        enc = {k: v.to(self.device) for k, v in enc.items()}
        return torch.softmax(self.model(**enc).logits, dim=-1).cpu().numpy()



In [ ]:
test = load_split('test')
model_dir = MODELS_DIR / 'mbert-sv'
predictor = CachedPredictor(model_dir)
y_pred = predictor.predict_proba(test['texto_modelo'].tolist()).argmax(axis=1)
y_true = test['label'].values
samples = []
for cls in range(4):
    idx_correct = np.where((y_true==cls) & (y_pred==cls))[0][:2]
    idx_error = np.where((y_true==cls) & (y_pred!=cls))[0][:1]
    samples.extend(idx_correct.tolist() + idx_error.tolist())
texts_sample = test.iloc[samples[:8]]['texto_modelo'].tolist()
print(f'Instancias seleccionadas: {len(texts_sample)}')


## LIME


In [ ]:
explainer = build_lime_explainer()
predict_proba_fn = make_predict_proba_fn(predictor.predict_proba)
lime_explanations = []
for text in texts_sample[:5]:
    exp = explain_instance(text, predict_proba_fn, explainer, num_samples=150)
    lime_explanations.append(exp)
    print(f'Clase predicha: {LABELS[exp["predicted_class"]]}')
    print('Top tokens:', exp['top_tokens'][:5])
    print('---')
global_lime = aggregate_lime_weights(lime_explanations)
global_lime


## SHAP


In [ ]:
shap_predict_fn = make_shap_predict_fn(predictor.model, predictor.tokenizer, predictor.device)
shap_texts = texts_sample[:3]
try:
    shap_values = explain_texts_shap(shap_texts, shap_predict_fn, tokenizer=predictor.tokenizer, max_evals=25)
    for i, text in enumerate(shap_texts):
        top = extract_top_tokens(shap_values, i, top_k=5)
        print(f'Texto {i}: {text[:60]}...')
        print('SHAP top:', top)
    shap_ok = True
except Exception as e:
    print(f'SHAP falló ({e}). LIME cubre el requisito.')
    shap_ok = False


## Sonda de sesgo dialectal


In [ ]:
# FP en No Tóxico con jerga
fp_mask = (y_true == 0) & (y_pred >= 1)
fp_with_jerga = test.iloc[np.where(fp_mask & (test['n_jerga']>0).values)[0]]
fp_without_jerga = test.iloc[np.where(fp_mask & (test['n_jerga']==0).values)[0]]
print(f'FP No Tóxico con jerga: {len(fp_with_jerga)}')
print(f'FP No Tóxico sin jerga: {len(fp_without_jerga)}')
if len(fp_with_jerga):
    print('Ejemplo FP con jerga:', fp_with_jerga['texto_modelo'].iloc[0][:100])


In [ ]:
# Contraste LIME vs SHAP (Jaccard top-k)
if shap_ok and lime_explanations:
    for i in range(min(3, len(lime_explanations))):
        lime_tokens = lime_explanations[i]['top_tokens']
        shap_top = extract_top_tokens(shap_values, i, top_k=10)
        shap_tokens = [t for t, _ in shap_top]
        j = jaccard_top_k(lime_tokens, shap_tokens)
        print(f'Instancia {i}: Jaccard LIME-SHAP = {j:.3f}')
print('Análisis de interpretabilidad completado.')
